# 03 — Baseline Model Evaluation

**Goal**: Establish a lower bound for performance using standard ML models (Logistic Regression and LightGBM) on the engineered feature set (AAC + DPC + Stats + 2-mers).

**Verify gates**:
- 5-fold Stratified CV AUC-ROC > 0.5
- Print mean ± std AUC
- Plot ROC curves
- Generate first submission file

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, auc, classification_report
from lightgbm import LGBMClassifier

plt.style.use('dark_background')
sns.set_palette('viridis')
print('Libraries loaded ✓')

## Step 3.1 — Load Features and Labels

In [ ]:
X_train = pd.read_pickle('../data/processed/X_train_features.pkl')
y_train = pd.read_pickle('../data/processed/y_train.pkl')
X_test  = pd.read_pickle('../data/processed/X_test_features.pkl')

print(f'Train features: {X_train.shape}')
print(f'Train labels:   {y_train.shape}')
print(f'Test features:  {X_test.shape}')

test_df  = pd.read_csv('../data/processed/test_clean.csv')

## Step 3.2 & 3.3 — 5-Fold Stratified CV (LogReg & LightGBM)

In [ ]:
def evaluate_model(model_name, model_fn, scale=False):
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    
    oof_preds = np.zeros(len(X_train))
    fold_metrics = []
    tprs = []
    mean_fpr = np.linspace(0, 1, 100)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train, y_train)):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_va, y_va = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        if scale:
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_va = scaler.transform(X_va)
            
        model = model_fn()
        model.fit(X_tr, y_tr)
        
        # Predict probabilities for AUC (+1 class)
        preds = model.predict_proba(X_va)[:, 1] 
        oof_preds[val_idx] = preds
        
        auc_score = roc_auc_score(y_va, preds)
        fold_metrics.append(auc_score)
        
        fpr, tpr, _ = roc_curve(y_va, preds)
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        
        ax.plot(fpr, tpr, lw=1, alpha=0.3, label=f'Fold {fold+1} (AUC = {auc_score:.3f})')

    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = auc(mean_fpr, mean_tpr)
    std_auc = np.std(fold_metrics)
    
    ax.plot(mean_fpr, mean_tpr, color='b', label=f'Mean ROC (AUC = {mean_auc:.3f} ± {std_auc:.3f})', lw=2)
    ax.plot([0, 1], [0, 1], linestyle='--', lw=2, color='r', alpha=0.8)
    
    ax.set_title(f'ROC Curve - {model_name}', fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(loc="lower right")
    plt.show()
    
    print(f"{model_name} CV Results: Mean AUC = {mean_auc:.4f} ± {std_auc:.4f}")
    return mean_auc

# 1. Logistic Regression (Requires scaling)
lr_fn = lambda: LogisticRegression(max_iter=1000, random_state=42, C=0.1, n_jobs=-1)
lr_auc = evaluate_model("Logistic Regression (L2)", lr_fn, scale=True)

# 2. LightGBM (Gradient Boosted Trees)
# Setting fixed seed and params for robust baseline
lgb_fn = lambda: LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_auc = evaluate_model("LightGBM", lgb_fn, scale=False)

## Step 3.6 — Generate Baseline Submission

In [ ]:
# Train LightGBM on FULL dataset for final submission
final_model = LGBMClassifier(
    n_estimators=300,        # Slightly more trees for full dataset
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

print('Training final LightGBM baseline on full train set...')
final_model.fit(X_train, y_train)

print('Predicting on test set...')
# We output probabilities of class 1 directly for ROC-AUC evaluation
test_preds = final_model.predict_proba(X_test)[:, 1]

# Create submission DataFrame
sub_df = pd.DataFrame({
    'ID': test_df['ID'],
    'Label': test_preds
})

sub_df.to_csv('../submissions/baseline_lgbm_v1.csv', index=False)
print(f'Saved baseline submission to submissions/baseline_lgbm_v1.csv! Shape: {sub_df.shape}')
display(sub_df.head())